## Parsing and injesting the sea_marks datafile


TODO:
- Support beacons by passing seamark type - unify repeated cod
- Support wrecks and rocks similarly

In [1]:
import json
import os
import time

os.chdir("../../..")

from tests.notebooks.graph_ingestion.neo4j_client import Neo4jClient

graph_client = Neo4jClient()
graph_client.delete_all_nodes()
graph_client.create_layers()

In [2]:
with open("raw_maps/seamarks_and_coastlines_solent.geojson") as map:
    raw_map = json.load(map)

conditional_processing = {
    "buoy_special_purpose": graph_client.injest_special_purpose_mark,
    "beacon_special_purpose": graph_client.injest_special_purpose_mark,
    "buoy_isolated_danger": graph_client.injest_isolated_danger_mark,
    "beacon_isolated_danger": graph_client.injest_isolated_danger_mark,
    "buoy_cardinal": graph_client.injest_cardinal_mark,
    "beacon_cardinal": graph_client.injest_cardinal_mark,
    "buoy_lateral": graph_client.injest_lateral_mark,
    "beacon_lateral": graph_client.injest_lateral_mark,
    "harbour": graph_client.injest_harbours,
    "wreck": graph_client.injest_wreck
}

features = raw_map["features"]
cleaned_features = []
mark_count = 0
coastline_count = 0

print(f"Attempting to insert {len(features)} new marks.")

unsupported_mark_types = []

for feature in features:  
    try:
        seamark_type = feature["properties"]["seamark:type"]
    except KeyError:
        pass
    else:
        try:
            feature_extraction = conditional_processing[seamark_type](feature)
            mark_count += 1
        except KeyError as e:
            if seamark_type not in unsupported_mark_types:
                unsupported_mark_types.append(seamark_type)
        except Exception as e:
            print(f"Failed to inject a mark: {e}")
            raise
    

    # Coastline geoms
    try:
        if feature["properties"]["natural"] == "coastline":
            graph_client.injest_coastline(feature)
            coastline_count += 1
    except KeyError:
        pass

print(f"Successfully inserted {mark_count} new marks.")
print(f"Successfully inserted {coastline_count} new coastlines")

print(f"The unsupported types are:\n{unsupported_mark_types}")

Attempting to insert 2068 new marks.
Successfully inserted 759 new marks.
Successfully inserted 379 new coastlines
The unsupported types are:
['landmark', 'building', 'rescue_station', 'light_major', 'platform', 'shoreline_construction', 'small_craft_facility', 'bridge', 'fairway', 'precautionary_area', 'recommended_track', 'navigation_line', 'causeway', 'sea_area', 'gate', 'cable_submarine', 'restricted_area', 'anchorage', 'notice', 'dumping_ground', 'pipeline_submarine', 'ferry_route', 'light_minor', 'obstruction', 'light_float', 'beacon_lateral', 'berth', 'beacon_cardinal', 'pile', 'crane', 'mooring', 'rock', 'pilot_boarding']


---

### Create the edges

In [3]:
start_time = time.time()
edges = graph_client.create_safe_edges()
end_time = time.time()

print(f"Successfully created {len(edges)} edges in {round(end_time-start_time,1)} seconds")

Successfully created 9228 edges in 204.0 seconds


---

#### Project spatial marks onto a Graph Data Science graph

In [4]:
graph_client.project_spatial_to_graph()

---
#### Delete all nodes

In [ ]:
graph_client.delete_all_nodes() 

---
##### A* route optimisation

In [5]:
import os

os.chdir("../../..")
from tests.notebooks.graph_ingestion.neo4j_client import Neo4jClient
graph_client = Neo4jClient()
graph_client.a_star_by_name(name_from="EC11", name_to="Hook") 

ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.shortestPath.astar.stream`: Caused by: java.lang.IllegalArgumentException: Relationship weight property `distance_km` not found in relationship types ['SAFE_EDGE']. Properties existing on all relationship types: []} {gql_status: 52N37} {gql_status_description: error: procedure exception - procedure execution error. Execution of the procedure gds.shortestPath.astar.stream() failed.}

___
### Testing

In [4]:
from shapely.geometry import LineString, Polygon

def line_intersects_polygon(p1, p2, polygon_coords):
    """
    p1, p2: (lon, lat)
    polygon_coords: [(lon, lat), (lon, lat), ...]
    """
    line = LineString([p1, p2])
    polygon = Polygon(polygon_coords)
    return line.intersects(polygon)


# -------------------------
# Example usage
# -------------------------

# North test mark
point_a = (-0.9927808, 50.7189197)
# South test mark
point_b = (-0.9929109, 50.7174626)

# North test danger zone
danger_polygon = [(-0.9909808, 50.7170297),
(-0.9910153864952742, 50.717380862579624),
(-0.9911178168414797, 50.71771853017825),
(-0.9912841546978555, 50.71802972641943),
(-0.9915080077938643, 50.718302492206135),
(-0.9917807735805647, 50.718526345302145),
(-0.9920919698217429, 50.718692683158515),
(-0.9924296374203709, 50.718795113504726),
(-0.9927808, 50.7188297),
(-0.9931319625796291, 50.718795113504726),
(-0.9934696301782572, 50.718692683158515),
(-0.9937808264194353, 50.718526345302145),
(-0.9940535922061358, 50.718302492206135),
(-0.9942774453021446, 50.71802972641943),
(-0.9944437831585203, 50.71771853017825),
(-0.9945462135047258, 50.717380862579624),
(-0.9945808, 50.7170297),
(-0.9945462135047258, 50.71667853742037),
(-0.9944437831585203, 50.71634086982174),
(-0.9942774453021446, 50.716029673580564),
(-0.9940535922061358, 50.71575690779386),
(-0.9937808264194353, 50.71553305469785),
(-0.9934696301782572, 50.71536671684148),
(-0.9931319625796291, 50.71526428649527),
(-0.9927808, 50.715229699999995),
(-0.9924296374203709, 50.71526428649527),
(-0.9920919698217429, 50.71536671684148),
(-0.9917807735805647, 50.71553305469785),
(-0.9915080077938643, 50.71575690779386),
(-0.9912841546978555, 50.716029673580564),
(-0.9911178168414797, 50.71634086982174),
(-0.9910153864952742, 50.71667853742037),
(-0.9909808, 50.7170297)]

print(line_intersects_polygon(point_a, point_b, danger_polygon))


True
